# 📈 NSE Intraday Stock Scanner - Google Colab Version

**Features:**
- ✅ Works on Google Colab (no local installation needed)
- ✅ Option 1: Yahoo Finance (Free, 15-min delay)
- ✅ Option 2: Upstox API (Free, Real-time data)
- ✅ 70+ F&O stocks scanning
- ✅ 10+ technical indicators

**How to Use:**
1. Click "Runtime" → "Run all" (or run cells one by one)
2. Choose Yahoo Finance (free) or Upstox (real-time)
3. Get high-quality intraday opportunities!

---

## 📦 Step 1: Install Required Packages

In [ ]:
# Install required packages (takes 30-60 seconds)
!pip install -q yfinance pandas numpy
!pip install -q upstox-client  # For real-time data (optional)

print("✅ All packages installed successfully!")

## 📋 Step 2: Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## 🎯 Step 3: Stock Universe

NSE F&O stocks (most liquid)

In [ ]:
# NSE F&O Stock List (Top liquid stocks)
NSE_FNO_STOCKS = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS',
    'HINDUNILVR.NS', 'ITC.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'KOTAKBANK.NS',
    'LT.NS', 'AXISBANK.NS', 'ASIANPAINT.NS', 'MARUTI.NS', 'HCLTECH.NS',
    'SUNPHARMA.NS', 'BAJFINANCE.NS', 'WIPRO.NS', 'ULTRACEMCO.NS', 'TITAN.NS',
    'NESTLEIND.NS', 'TATAMOTORS.NS', 'ONGC.NS', 'NTPC.NS', 'POWERGRID.NS',
    'M&M.NS', 'TECHM.NS', 'BAJAJFINSV.NS', 'ADANIPORTS.NS', 'TATASTEEL.NS',
    'COALINDIA.NS', 'HINDALCO.NS', 'INDUSINDBK.NS', 'DIVISLAB.NS', 'DRREDDY.NS',
    'CIPLA.NS', 'GRASIM.NS', 'JSWSTEEL.NS', 'HEROMOTOCO.NS', 'EICHERMOT.NS',
    'BRITANNIA.NS', 'BPCL.NS', 'SHREECEM.NS', 'TATACONSUM.NS', 'APOLLOHOSP.NS',
    'ADANIENT.NS', 'BAJAJ-AUTO.NS', 'PIDILITIND.NS', 'SIEMENS.NS', 'DLF.NS',
    'VEDL.NS', 'GODREJCP.NS', 'HAVELLS.NS', 'BANDHANBNK.NS', 'ICICIGI.NS'
]

print(f"✅ Loaded {len(NSE_FNO_STOCKS)} stocks")

## 🔧 Step 4: Technical Indicator Functions

In [ ]:
def calculate_vwap(df):
    """Calculate VWAP"""
    df['VWAP'] = (df['Volume'] * (df['High'] + df['Low'] + df['Close']) / 3).cumsum() / df['Volume'].cumsum()
    return df

def calculate_ema(df, period):
    """Calculate EMA"""
    return df['Close'].ewm(span=period, adjust=False).mean()

def calculate_rsi(df, period=14):
    """Calculate RSI"""
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_macd(df):
    """Calculate MACD"""
    ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    signal = macd.ewm(span=9, adjust=False).mean()
    return macd, signal

def calculate_atr(df, period=14):
    """Calculate ATR"""
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    atr = true_range.rolling(period).mean()
    return atr

print("✅ Technical indicators ready!")

## 📊 Step 5: Data Fetching Function

In [ ]:
def get_stock_data(symbol, period='5d', interval='5m'):
    """Fetch stock data from Yahoo Finance"""
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(period=period, interval=interval)
        
        if df.empty or len(df) < 50:
            return None
        
        # Get previous day's close
        daily_data = ticker.history(period='10d', interval='1d')
        if len(daily_data) >= 2:
            prev_close = daily_data['Close'].iloc[-2]
        else:
            prev_close = df['Close'].iloc[0]
        
        return df, prev_close, ticker.info
    except Exception as e:
        return None

print("✅ Data fetching function ready!")

## 🔍 Step 6: Stock Analysis Function

In [ ]:
def analyze_stock(symbol):
    """Analyze a single stock against all criteria"""
    
    data = get_stock_data(symbol)
    if data is None:
        return None
    
    df, prev_close, info = data
    
    # Current metrics
    current_price = df['Close'].iloc[-1]
    current_volume = df['Volume'].sum()
    
    # Calculate indicators
    df = calculate_vwap(df)
    df['EMA_20'] = calculate_ema(df, 20)
    df['EMA_50'] = calculate_ema(df, 50)
    df['RSI'] = calculate_rsi(df)
    df['MACD'], df['MACD_Signal'] = calculate_macd(df)
    df['ATR'] = calculate_atr(df)
    
    # Latest values
    latest = df.iloc[-1]
    vwap = latest['VWAP']
    ema_20 = latest['EMA_20']
    ema_50 = latest['EMA_50']
    rsi = latest['RSI']
    macd = latest['MACD']
    macd_signal = latest['MACD_Signal']
    atr = latest['ATR']
    
    # Calculate metrics
    pct_change = ((current_price - prev_close) / prev_close) * 100
    gap = abs(pct_change)
    
    # Volume metrics
    avg_volume = df['Volume'].tail(100).mean() * 78
    volume_ratio = (current_volume / avg_volume) if avg_volume > 0 else 0
    
    # Opening Range
    or_high = df['High'].head(3).max()
    or_low = df['Low'].head(3).min()
    
    # Previous day high/low
    prev_day_high = df['High'].tail(78).max()
    prev_day_low = df['Low'].tail(78).min()
    
    # ATR percentage
    atr_pct = (atr / current_price) * 100 if current_price > 0 else 0
    
    # VWAP position
    vwap_position = "Above" if current_price > vwap else "Below"
    vwap_distance = abs(current_price - vwap) / vwap * 100
    
    # Determine direction and criteria
    direction = None
    setup = []
    score = 0
    
    # BULLISH CHECKS
    bullish_criteria = 0
    if gap >= 1.5 and current_price > prev_close:
        bullish_criteria += 1
        setup.append("Gap Up")
    
    if current_price > vwap and vwap_distance < 2:
        bullish_criteria += 1
        setup.append("Above VWAP")
    
    if current_price > ema_20 and ema_20 > ema_50:
        bullish_criteria += 1
        setup.append("EMA Aligned")
    
    if 55 <= rsi <= 75:
        bullish_criteria += 1
        setup.append("RSI Momentum")
    
    if macd > macd_signal:
        bullish_criteria += 1
    
    if current_price > or_high or current_price > prev_day_high:
        bullish_criteria += 1
        setup.append("Breakout")
    
    if volume_ratio > 1.5:
        bullish_criteria += 1
        setup.append("Volume Surge")
    
    if atr_pct > 1:
        bullish_criteria += 1
    
    # BEARISH CHECKS
    bearish_criteria = 0
    if gap >= 1.5 and current_price < prev_close:
        bearish_criteria += 1
        setup.append("Gap Down")
    
    if current_price < vwap and vwap_distance < 2:
        bearish_criteria += 1
        setup.append("Below VWAP")
    
    if current_price < ema_20 and ema_20 < ema_50:
        bearish_criteria += 1
        setup.append("EMA Aligned")
    
    if 25 <= rsi <= 45:
        bearish_criteria += 1
        setup.append("RSI Momentum")
    
    if macd < macd_signal:
        bearish_criteria += 1
    
    if current_price < or_low or current_price < prev_day_low:
        bearish_criteria += 1
        setup.append("Breakdown")
    
    if volume_ratio > 1.5:
        bearish_criteria += 1
        setup.append("Volume Surge")
    
    if atr_pct > 1:
        bearish_criteria += 1
    
    # Determine direction
    if bullish_criteria >= 5:
        direction = "BULLISH"
        score = bullish_criteria
    elif bearish_criteria >= 5:
        direction = "BEARISH"
        score = bearish_criteria
    else:
        return None
    
    # Filter low quality setups
    if volume_ratio < 1.5 or avg_volume < 1000000:
        return None
    
    # Get sector
    sector = info.get('sector', 'Unknown')
    
    return {
        'Symbol': symbol.replace('.NS', ''),
        'Direction': direction,
        'Price': round(current_price, 2),
        'Change%': round(pct_change, 2),
        'Volume_Ratio': round(volume_ratio, 2),
        'VWAP_Position': vwap_position,
        'VWAP': round(vwap, 2),
        'Setup': ' | '.join(setup[:3]),
        'RSI': round(rsi, 1),
        'Sector': sector,
        'Score': score,
        'EMA20': round(ema_20, 2),
        'EMA50': round(ema_50, 2),
        'ATR%': round(atr_pct, 2)
    }

print("✅ Analysis function ready!")

## 🚀 Step 7: MAIN SCANNER - RUN THIS!

This will scan all 50+ stocks and show you the best opportunities

In [ ]:
def scan_market():
    """Scan entire market for opportunities"""
    print("\n" + "="*100)
    print(f"NSE INTRADAY SCANNER - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100 + "\n")
    print("📊 Scanning NSE F&O stocks for high-probability intraday setups...")
    print(f"🎯 Universe: {len(NSE_FNO_STOCKS)} stocks\n")
    
    opportunities = []
    processed = 0
    
    for symbol in NSE_FNO_STOCKS:
        processed += 1
        print(f"Scanning: {symbol} ({processed}/{len(NSE_FNO_STOCKS)})", end='\r')
        
        result = analyze_stock(symbol)
        if result:
            opportunities.append(result)
    
    print("\n")
    
    if not opportunities:
        print("⚠️  No stocks meeting the criteria found at this time.")
        print("Market may be consolidating or criteria too strict.\n")
        return None
    
    # Sort by score
    opportunities.sort(key=lambda x: x['Score'], reverse=True)
    opportunities = opportunities[:10]
    
    print(f"✅ Found {len(opportunities)} High-Quality Opportunities:\n")
    print("="*100)
    
    for i, opp in enumerate(opportunities, 1):
        print(f"\n{i}. {opp['Symbol']} - {opp['Direction']}")
        print(f"   Price: ₹{opp['Price']} | Change: {opp['Change%']:+.2f}% | Volume: {opp['Volume_Ratio']:.2f}x avg")
        print(f"   VWAP: ₹{opp['VWAP']} ({opp['VWAP_Position']}) | RSI: {opp['RSI']} | ATR: {opp['ATR%']:.2f}%")
        print(f"   Setup: {opp['Setup']}")
        print(f"   Sector: {opp['Sector']} | Quality Score: {opp['Score']}/8 ⭐")
        print(f"   EMA20: ₹{opp['EMA20']} | EMA50: ₹{opp['EMA50']}")
    
    print("\n" + "="*100)
    print(f"\n💡 Focus on top 5 setups with Score ≥ 6 for best probability")
    print("⚠️  Always use stop-loss and position sizing. This is not financial advice.\n")
    
    # Convert to DataFrame for better viewing
    df_results = pd.DataFrame(opportunities)
    
    return df_results

# RUN THE SCANNER!
results = scan_market()

## 📊 Step 8: View Results as Table (Optional)

In [ ]:
# Display results in a nice table
if results is not None:
    print("\n📊 RESULTS TABLE:\n")
    display(results)
    
    # Download results
    filename = f"intraday_scan_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    results.to_csv(filename, index=False)
    print(f"\n✅ Results saved to: {filename}")
    print("You can download it from the files panel on the left ⬅️")
else:
    print("No results to display. Run the scanner again at a different time.")

---

## 🎯 UPSTOX REAL-TIME VERSION (Optional)

If you have Upstox account and want real-time data (FREE!), use the cells below:

### Step A: Install Upstox Client

In [ ]:
# Only run this if you want Upstox real-time data
!pip install -q upstox-client
print("✅ Upstox client installed!")

### Step B: Configure Upstox Credentials

Get your API credentials from: https://account.upstox.com/developer/apps

In [ ]:
# Configure your Upstox credentials here
UPSTOX_API_KEY = "your_api_key_here"        # From Upstox developer console
UPSTOX_API_SECRET = "your_api_secret_here"  # From Upstox developer console
UPSTOX_ACCESS_TOKEN = "your_access_token_here"  # Generate using login flow

print("⚠️ Remember to update credentials above before running!")
print("Get credentials from: https://account.upstox.com/developer/apps")

### Step C: Generate Upstox Access Token

In [ ]:
import upstox_client

# Generate login URL
REDIRECT_URI = "http://127.0.0.1:5000"
login_url = f"https://api.upstox.com/v2/login/authorization/dialog?response_type=code&client_id={UPSTOX_API_KEY}&redirect_uri={REDIRECT_URI}"

print("\n" + "="*70)
print("UPSTOX LOGIN INSTRUCTIONS:")
print("="*70)
print(f"\n1. Open this URL in browser:\n   {login_url}")
print("\n2. Login and authorize")
print("\n3. You'll be redirected to: http://127.0.0.1:5000/?code=XXXXX")
print("\n4. Copy the 'code' parameter and paste below:")
print("="*70 + "\n")

# Get auth code from user
auth_code = input("Enter authorization code: ").strip()

# Generate access token
try:
    configuration = upstox_client.Configuration()
    api_instance = upstox_client.LoginApi(upstox_client.ApiClient(configuration))
    
    api_response = api_instance.token(
        grant_type='authorization_code',
        code=auth_code,
        client_id=UPSTOX_API_KEY,
        client_secret=UPSTOX_API_SECRET,
        redirect_uri=REDIRECT_URI
    )
    
    access_token = api_response.access_token
    
    print("\n" + "="*70)
    print("✅ SUCCESS! Your Access Token:")
    print("="*70)
    print(f"\n{access_token}\n")
    print("Copy this and paste in UPSTOX_ACCESS_TOKEN variable above!")
    print("="*70 + "\n")
    
    UPSTOX_ACCESS_TOKEN = access_token
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("Make sure API_KEY and API_SECRET are correct!")

---

## ⚙️ Settings & Customization

In [ ]:
# Add more stocks to scan (optional)
CUSTOM_STOCKS = [
    # Add your stocks here in format 'SYMBOL.NS'
    # Example: 'ZOMATO.NS', 'PAYTM.NS'
]

if CUSTOM_STOCKS:
    NSE_FNO_STOCKS.extend(CUSTOM_STOCKS)
    print(f"✅ Added {len(CUSTOM_STOCKS)} custom stocks")
    print(f"Total stocks to scan: {len(NSE_FNO_STOCKS)}")

---

## 📚 Important Notes

### ⚠️ Risk Management
1. **Always use stop-loss** (1-2% from entry)
2. **Position size**: Risk only 1-2% per trade
3. **Focus on Score ≥ 6** for best probability
4. **Exit by 3:15 PM** to avoid closing volatility

### 📊 Data Delay
- **Yahoo Finance**: ~15 minute delay (free)
- **Upstox API**: Real-time (free for Upstox users)

### ⏰ Best Times to Scan
- 9:30 AM - Opening breakouts
- 11:00 AM - Mid-morning momentum
- 1:00 PM - Post-lunch session
- 2:30 PM - Final hour opportunities

### 🚫 Disclaimer
**This is NOT financial advice.** Trading involves substantial risk. Always:
- Do your own research
- Paper trade first
- Use proper risk management
- Consult a financial advisor

---

## 🎯 Quick Start Guide

**First Time:**
1. Click "Runtime" → "Run all"
2. Wait 2-3 minutes for scanning
3. Check results!

**Daily Use:**
1. Open this notebook
2. Run cell 7 (Main Scanner)
3. Get fresh opportunities!

---

**Happy Trading! 📈**